<a href="https://colab.research.google.com/github/mvinegret/ecommerce_AB_Test_SQL_Python/blob/main/E-commerce%20AB%20Test%20Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import auth, files
from google.cloud import bigquery
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import pearsonr, mannwhitneyu, shapiro, kruskal
from statsmodels.stats.proportion import proportions_ztest
import warnings
warnings.filterwarnings('ignore', message='invalid value encountered in scalar divide')

In [2]:
auth.authenticate_user()
client = bigquery.Client(project='data-analytics-mate')

In [3]:
query = '''
WITH
  session_info AS (
    SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
    FROM `DA.ab_test` ab
    JOIN `DA.session` s
      ON ab.ga_session_id = s.ga_session_id
    JOIN `DA.session_params` sp
      ON s.ga_session_id = sp.ga_session_id
  ),
  sessions_with_orders AS (
    SELECT
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT o.ga_session_id) AS session_with_orders
    FROM `DA.order` o
    JOIN session_info
      ON o.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  ),
  events AS (
    SELECT
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name,
      COUNT(ep.ga_session_id) AS event_cnt
    FROM `DA.event_params` ep
    JOIN session_info
      ON ep.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name
  ),
  session AS (
    SELECT
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
    FROM session_info
    GROUP BY
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  ),
  account AS (
    SELECT
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
    FROM `DA.account_session` acs
    JOIN session_info
      ON acs.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  )
SELECT
  sessions_with_orders.date,
  sessions_with_orders.ga_session_id,
  sessions_with_orders.country,
  sessions_with_orders.device,
  sessions_with_orders.continent,
  sessions_with_orders.channel,
  sessions_with_orders.test,
  sessions_with_orders.test_group,
  'session with orders' AS event_name,
  sessions_with_orders.session_with_orders AS value
FROM sessions_with_orders
UNION ALL
SELECT
  events.date,
  events.ga_session_id,
  events.country,
  events.device,
  events.continent,
  events.channel,
  events.test,
  events.test_group,
  event_name,
  event_cnt AS value
FROM events
UNION ALL
SELECT
  session.date,
  session.ga_session_id,
  session.country,
  session.device,
  session.continent,
  session.channel,
  session.test,
  session.test_group,
  'session' AS event_name,
  session_cnt AS value
FROM session
UNION ALL
SELECT
  account.date,
  account.ga_session_id,
  account.country,
  account.device,
  account.continent,
  account.channel,
  account.test,
  account.test_group,
  'new account' AS event_name,
  new_account_cnt AS value
FROM account

'''

query_job = client.query(query)
output = query_job.result()

In [4]:
df = output.to_dataframe()
df.head()

,date,ga_session_id,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,2824251388,Peru,mobile,Americas,Organic Search,2,1,session with orders,1
1,2020-11-02,3382486655,Pakistan,mobile,Asia,Direct,2,2,session with orders,1
2,2020-11-03,9308371575,Peru,desktop,Americas,Paid Search,2,2,session with orders,1
3,2020-11-08,5006344402,Denmark,mobile,Europe,Social Search,2,2,session with orders,1
4,2020-11-10,7952273605,Slovakia,desktop,Europe,Paid Search,2,1,session with orders,1


In [5]:
# Fixing event names
df['event_name'] = df['event_name'].str.strip().str.lower().str.replace(" ", "_")
df.head()

,date,ga_session_id,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,2824251388,Peru,mobile,Americas,Organic Search,2,1,session_with_orders,1
1,2020-11-02,3382486655,Pakistan,mobile,Asia,Direct,2,2,session_with_orders,1
2,2020-11-03,9308371575,Peru,desktop,Americas,Paid Search,2,2,session_with_orders,1
3,2020-11-08,5006344402,Denmark,mobile,Europe,Social Search,2,2,session_with_orders,1
4,2020-11-10,7952273605,Slovakia,desktop,Europe,Paid Search,2,1,session_with_orders,1


In [6]:
test_metrics = ["add_payment_info", "add_shipping_info", "begin_checkout", "new_account", "session"]
df_filtered = df[df['event_name'].isin(test_metrics)]

pivot_table_metrics = pd.pivot_table(
    data = df_filtered,
    values= "value",
    index = ["test", "test_group", "continent", "device"],
    columns = "event_name",
    aggfunc = "sum",
    fill_value=0
).reset_index()

display(pivot_table_metrics)

event_name,test,test_group,continent,device,add_payment_info,add_shipping_info,begin_checkout,new_account,session
0,1,1,(not set),desktop,0,1,1,6,55
1,1,1,(not set),mobile,7,3,3,0,39
2,1,1,(not set),tablet,0,0,0,1,3
3,1,1,Africa,desktop,12,16,20,18,285
4,1,1,Africa,mobile,7,25,34,16,198
...,...,...,...,...,...,...,...,...,...
139,4,2,Europe,mobile,259,359,891,630,7606
140,4,2,Europe,tablet,13,27,68,49,478
141,4,2,Oceania,desktop,21,39,89,51,629
142,4,2,Oceania,mobile,17,20,48,36,408


In [7]:
# Define the metrics we're testing (excludes 'session' which is the denominator)
metrics = ["add_payment_info", "add_shipping_info", "begin_checkout", "new_account"]

# Get unique values for breakdowns
continents = pivot_table_metrics["continent"].unique()
device = pivot_table_metrics["device"].unique()

# Initialize empty list to store all test results
ab_results = []

# ==========================================
# PART 1: CALCULATE OVERALL RESULTS (ALL REGIONS, ALL DEVICES)
# ==========================================

# Loop through each test (test 1, test 2, etc.)
for test_id in pivot_table_metrics['test'].unique():
    # Filter data for current test
    current_test = pivot_table_metrics[pivot_table_metrics['test'] == test_id]

    # Sum all metrics for each test group across all segments (continents/devices)
    group_A_total = current_test[current_test["test_group"] == 1].sum(numeric_only=True)  # Control group
    group_B_total = current_test[current_test["test_group"] == 2].sum(numeric_only=True)  # Test group

    # Test each metric
    for metric in metrics:
        # Get numerators (event counts) and denominators (session counts)
        num_c, num_t = group_A_total[metric], group_B_total[metric]  # c = control, t = test
        den_c, den_t = group_A_total["session"], group_B_total["session"]

        # Only proceed if both groups have sessions (avoid division by zero)
        if den_c > 0 and den_t > 0:
            # Calculate conversion rates
            conversion_test = num_t / den_t
            conversion_control = num_c / den_c

            # Calculate percentage change (lift)
            if conversion_control > 0:
                metric_change_pct = round(((conversion_test / conversion_control) - 1) * 100, 2)
            else:
                metric_change_pct = 0  # If control is 0, can't calculate meaningful lift

            # Run two-proportion z-test to check statistical significance
            z_stat, p_value = proportions_ztest(count=[num_t, num_c], nobs=[den_t, den_c])

            # Store results for overall analysis
            ab_results.append({
                'test_number': test_id,
                'continent': 'All Regions',  # Indicates this is aggregated data
                'device': 'All Devices',     # Indicates this is aggregated data
                'metric': metric,
                'numerator_test': num_t,
                'denominator_test': den_t,
                'conversion_test': conversion_test,
                'numerator_control': num_c,
                'denominator_control': den_c,
                'conversion_control': conversion_control,
                'metric_change_pct': metric_change_pct,
                'p_value': round(p_value, 4),
                'z_score': round(z_stat, 4),
                'significant': p_value < 0.05  # True if p-value < 0.05
            })

# ==========================================
# PART 2: CALCULATE BREAKDOWN RESULTS (BY CONTINENT AND DEVICE)
# ==========================================

for test_id in pivot_table_metrics['test'].unique():
    for cont in continents:
        for dev in device:
            # Filter data for this specific combination
            current_slice = pivot_table_metrics[
                (pivot_table_metrics['test'] == test_id) &
                (pivot_table_metrics['continent'] == cont) &
                (pivot_table_metrics['device'] == dev)
            ]

            # Split into control (group A) and test (group B)
            gA = current_slice[current_slice["test_group"] == 1]
            gB = current_slice[current_slice["test_group"] == 2]

            # Only proceed if both groups have data for this segment
            if not gA.empty and not gB.empty:
                # Test each metric
                for metric in metrics:
                    # Extract values (using .iloc[0] since each slice should have 1 row)
                    num_c, num_t = gA[metric].iloc[0], gB[metric].iloc[0]
                    den_c, den_t = gA["session"].iloc[0], gB["session"].iloc[0]

                    # Only test segments with sufficient sample size (> 30 sessions per group)
                    # This avoids false positives from small samples
                    if den_c > 30 and den_t > 30:
                        # Calculate conversion rates
                        conversion_test = num_t / den_t
                        conversion_control = num_c / den_c

                        # Calculate percentage change
                        if conversion_control > 0:
                            metric_change_pct = round(((conversion_test / conversion_control) - 1) * 100, 2)
                        else:
                            metric_change_pct = 0

                        # Run statistical test
                        z_stat, p_value = proportions_ztest(count=[num_t, num_c], nobs=[den_t, den_c])

                        # Store breakdown results
                        ab_results.append({
                            'test_number': test_id,
                            'continent': cont,
                            'device': dev,
                            'metric': metric,
                            'numerator_test': num_t,
                            'denominator_test': den_t,
                            'conversion_test': conversion_test,
                            'numerator_control': num_c,
                            'denominator_control': den_c,
                            'conversion_control': conversion_control,
                            'metric_change_pct': metric_change_pct,
                            'p_value': round(p_value, 4),
                            'z_score': round(z_stat, 4),
                            'significant': p_value < 0.05
                        })

# ==========================================
# PART 3: CREATE FINAL DATAFRAME AND EXPORT
# ==========================================

# Convert results list to DataFrame
final_df = pd.DataFrame(ab_results)

# Clean up missing continent labels
final_df['continent'] = final_df['continent'].replace('(not set)', 'Unknown')

# Export to CSV and download
final_df.to_csv('ab_test_final_master.csv', index=False)
files.download('ab_test_final_master.csv')

display(final_df.head())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,test_number,continent,device,metric,numerator_test,denominator_test,conversion_test,numerator_control,denominator_control,conversion_control,metric_change_pct,p_value,z_score,significant
0,1,All Regions,All Devices,add_payment_info,2229,45193,0.049322,1988,45362,0.043825,12.54,0.0001,3.9249,True
1,1,All Regions,All Devices,add_shipping_info,3221,45193,0.071272,3034,45362,0.066884,6.56,0.0092,2.6036,True
2,1,All Regions,All Devices,begin_checkout,4021,45193,0.088974,3784,45362,0.083418,6.66,0.0029,2.9788,True
3,1,All Regions,All Devices,new_account,3681,45193,0.081451,3823,45362,0.084278,-3.35,0.1229,-1.5429,False
4,2,All Regions,All Devices,add_payment_info,2409,50244,0.047946,2344,50637,0.046290,3.58,0.2146,1.2410,False


## **AB Test Results - Overall Conclusions**

### **1. Test Performance Summary**

**Test 1**
- **3 out of 4 metrics showed significant positive improvements**
- Best performer across all tests with strong lifts in key conversion funnel metrics:
  - `add_payment_info`: **+12.54%** (p=0.0001), highly statistically significant
  - `begin_checkout`: **+6.66%** (p=0.0029), statistically significant
  - `add_shipping_info`: **+6.56%** (p=0.0092), statistically significant
- Only `new_account` showed a non-significant decrease (-3.35%, p=0.1229)
- **Recommendation:** Test 1 should be rolled out to production

**Test 2**
- **0 out of 4 metrics were statistically significant**
- All metrics showed marginal positive lifts (+1.24% to +3.58%) but none reached significance
- **Recommendation:** Discard this variant, changes had no measurable effect

**Test 3**
- **1 out of 4 metrics significant, but in the wrong direction**
- `begin_checkout` significantly **decreased by -3.35%** (p=0.0120)
- Other metrics showed no significant changes
- **Recommendation:** Do not implement, this variant hurt checkout conversion

**Test 4**
- **2 out of 4 metrics significant, both negative**
- `begin_checkout`: **-2.35%** (p=0.0459), significant decrease
- `new_account`: **-3.36%** (p=0.0175), significant decrease
- **Recommendation:** Do not implement, this variant actively harmed performance

---

### **2. Segment-Level Insights**

**Geographic Performance:**
- **Europe** showed the highest % of significant results (37.5% of tests), suggesting European users may be more sensitive to UI/UX changes
- **Americas and Africa** both showed 18.8% significance rates, changes had moderate impact
- **Asia** at 22.9%, middle performer
- **Unknown** continent showed 50% significance, but this is likely due to small sample sizes and should be interpreted cautiously

**Device Performance:**
- **Desktop** users showed the most significant results (32.3%), indicating desktop experience changes had the strongest measurable impact
- **Mobile** followed at 28.1%, still substantial but slightly less pronounced
- **Tablet** showed the lowest rate at 25%, either smallest sample sizes or tablet users are less affected by changes

---

### **3. Recommendations**

1. **Implement Test 1 immediately**, it's a proven winner with strong statistical backing
2. **Do not implement Tests 2, 3, or 4**, they either had no effect or actively hurt performance
3. **Investigate the account creation issue**, why did multiple tests negatively impact `new_account`? Consider running a dedicated test focused on improving registration UX
4. **Consider regional rollouts**, given Europe's higher sensitivity to changes, consider testing future variants there first before global rollout
5. **Focus on mobile optimization next**, mobile showed slightly lower significance rates, suggesting room for improvement in mobile-specific UX

## Useful links
[Tableau Dashboard](https://public.tableau.com/views/ABTestDashboard_17717866785340/ABTest?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)  
[CSV output (Drive link)](https://drive.google.com/file/d/1ZiYnRU630h9UK75EYdHs9LFTeBO3murX/view?usp=sharing)